# DE2 — Assignment 3: Graphs or Clustering
> Author : Badr TAJINI - Data Engineering II (Data-Intensive Workloads) - ESIEE 2025-2026

**Track:** A — Esport (CS:GO Match Results)

**Path chosen:** Graph Processing

**Names:** Student 1 — Student 2

---
### Graph Description

We build a **team interaction graph** from CS:GO match data:
- **Vertices**: Teams (unique team names)
- **Edges**: Matches between teams (weighted by frequency)

### Pipeline
1. Build Graph (vertices + edges)
2. Run PageRank algorithm
3. Compare partitioning strategies
4. Analyze convergence
5. Save evidence & metrics

## 0. Setup

In [1]:
import os, time, pathlib, json
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window

# Network config
DE2_SPARK_DRIVER_HOST  = os.environ.get('DE2_SPARK_DRIVER_HOST',  '127.0.0.1')
DE2_SPARK_BIND_ADDRESS = os.environ.get('DE2_SPARK_BIND_ADDRESS', '0.0.0.0')
os.environ.setdefault('SPARK_LOCAL_IP', DE2_SPARK_DRIVER_HOST)

# Paths
NOTEBOOK_DIR = pathlib.Path(os.getcwd())
SOURCE_CSV = NOTEBOOK_DIR / 'data' / 'TrackA_CSGO' / 'results.csv'
OUT_DIR = NOTEBOOK_DIR / 'outputs' / 'lab3'
PROOF_DIR = NOTEBOOK_DIR / 'proof'
METRICS_LOG = NOTEBOOK_DIR / 'lab3_metrics_log.csv'

for d in [OUT_DIR, PROOF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print('Spark version:', spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        print('Spark UI:', ui_url)
        print('Spark UI (WSL/Windows):', f'http://localhost:{ui_port}')

spark = (SparkSession.builder
    .appName('de2-assignment3')
    .master('local[*]')

    .config('spark.driver.memory', '6g')

    .config('spark.executor.memory', '6g')

    .config('spark.driver.maxResultSize', '2g')
    .config('spark.driver.host', DE2_SPARK_DRIVER_HOST)
    .config('spark.driver.bindAddress', DE2_SPARK_BIND_ADDRESS)
    .config('spark.ui.bindAddress', DE2_SPARK_BIND_ADDRESS)
    .config('spark.sql.shuffle.partitions', '8')
    .getOrCreate())

show_spark_ui(spark)
print(f'\nOutputs: {OUT_DIR}')

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/florian/.local/lib/python3.12/site-packages/traitlets/config/appl

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.




A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/florian/.local/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/home/florian/.lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



26/05/11 21:36:02 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Spark UI: http://127.0.0.1:4040
Spark UI (WSL/Windows): http://localhost:4040

Outputs: /mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/outputs/lab3


## Step 1 — Data & Features: Build Graph

In [2]:
# Load match data
raw_schema = StructType([
    StructField('date', StringType(), True),
    StructField('team_1', StringType(), True),
    StructField('team_2', StringType(), True),
    StructField('_map', StringType(), True),
    StructField('result_1', IntegerType(), True),
    StructField('result_2', IntegerType(), True),
    StructField('map_winner', IntegerType(), True),
    StructField('match_id', IntegerType(), True),
])

matches_df = (spark.read.schema(raw_schema).option('header', 'true')
    .csv(str(SOURCE_CSV))
    .filter(F.col('team_1').isNotNull() & F.col('team_2').isNotNull()))

print(f'Matches loaded: {matches_df.count():,}')
matches_df.show(5)

Matches loaded: 45,773
+----------+-------------------+--------------+-------+--------+--------+----------+--------+
|      date|             team_1|        team_2|   _map|result_1|result_2|map_winner|match_id|
+----------+-------------------+--------------+-------+--------+--------+----------+--------+
|2020-03-18|            Recon 5|       TeamOne|  Dust2|       0|      16|         2|       2|
|2020-03-18|            Recon 5|       TeamOne|Inferno|      13|      16|         2|       2|
|2020-03-18|New England Whalers|      Station7|Inferno|      12|      16|         2|       1|
|2020-03-18|            Rugratz|Bad News Bears|Inferno|       7|      16|         2|       2|
|2020-03-18|            Rugratz|Bad News Bears|Vertigo|       8|      16|         2|       2|
+----------+-------------------+--------------+-------+--------+--------+----------+--------+
only showing top 5 rows


26/05/11 21:36:07 WARN CSVHeaderChecker: Number of column in CSV header is not equal to number of fields in the schema:
 Header length: 19, schema size: 8
CSV file: file:///mnt/c/Users/flori/OneDrive/Documents/Nouvelle_Sauvegarde/ESIEE/E4FD/Data_engineering_TAJINI/DE2/data/TrackA_CSGO/results.csv


In [3]:
# Build vertices (unique teams)
teams_1 = matches_df.select(F.col('team_1').alias('team'))
teams_2 = matches_df.select(F.col('team_2').alias('team'))
unique_teams = teams_1.union(teams_2).distinct().filter(F.col('team').isNotNull())

vertices = (unique_teams
    .withColumn('id', F.row_number().over(Window.orderBy('team')) - 1)
    .select(F.col('id').cast(LongType()), F.col('team').alias('name')))

vertices.cache()
n_vertices = vertices.count()
print(f'Vertices: {n_vertices:,} teams')
vertices.orderBy('id').show(10)

26/05/11 21:36:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 21:36:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 21:36:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 21:36:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 21:36:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 21:36:08 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/11 2

Vertices: 1,554 teams
+---+-----------+
| id|       name|
+---+-----------+
|  0|100 Thieves|
|  1|100pinggods|
|  2|       1337|
|  3| 1337HUANIA|
|  4|  13th Hour|
|  5|       1944|
|  6|        1UP|
|  7|       1WIN|
|  8|       2019|
|  9|    20matar|
+---+-----------+
only showing top 10 rows


In [4]:
# Build edges (matches between teams)
edges_raw = (matches_df
    .join(vertices.withColumnRenamed('id', 'src').withColumnRenamed('name', 'team_1'), on='team_1')
    .join(vertices.withColumnRenamed('id', 'dst').withColumnRenamed('name', 'team_2'), on='team_2')
    .select('src', 'dst'))

# Normalize edges (undirected)
edges_norm = (edges_raw
    .withColumn('src_norm', F.least('src', 'dst'))
    .withColumn('dst_norm', F.greatest('src', 'dst'))
    .select(F.col('src_norm').alias('src'), F.col('dst_norm').alias('dst')))

edges = (edges_norm.groupBy('src', 'dst')
    .agg(F.count('*').alias('weight'))
    .filter(F.col('src') != F.col('dst')))

edges.cache()
n_edges = edges.count()
print(f'Edges: {n_edges:,} team pairs')
print(f'Density: {n_edges / (n_vertices * (n_vertices - 1) / 2):.4f}')
edges.orderBy(F.desc('weight')).show(10)

Edges: 12,392 team pairs
Density: 0.0103
+---+----+------+
|src| dst|weight|
+---+----+------+
|120| 686|    75|
|120| 417|    73|
|242| 833|    69|
|417| 801|    66|
|462|1460|    66|
|120| 801|    65|
|417|1502|    63|
|242| 517|    63|
|120|1460|    61|
|686| 779|    59|
+---+----+------+
only showing top 10 rows


## Step 2 — Iterative Algorithm: PageRank

In [5]:
def run_pagerank(vertices_df, edges_df, max_iter=20, damping=0.85, tol=0.0001, partition_strategy='default'):
    n = vertices_df.count()
    if partition_strategy == 'custom':
        edges_df = edges_df.repartition(8, 'src')
        print('Applied custom partitioning: repartition(8, src)')
    
    ranks = vertices_df.select(F.col('id'), F.lit(1.0 / n).alias('rank'))
    out_degrees = edges_df.groupBy('src').agg(F.sum('weight').alias('out_degree'))
    metrics = []
    
    for iteration in range(max_iter):
        iter_start = time.time()
        contribs = (edges_df.join(ranks.withColumnRenamed('id', 'src'), on='src')
            .join(out_degrees, on='src')
            .select(F.col('dst').alias('id'),
                   (F.col('rank') * F.col('weight') / F.col('out_degree')).alias('contrib'))
            .groupBy('id').agg(F.sum('contrib').alias('contrib')))
        
        new_ranks = (vertices_df.select('id').join(contribs, on='id', how='left')
            .select(F.col('id'),
                   (F.lit(1 - damping) / n + damping * F.coalesce(F.col('contrib'), F.lit(0.0))).alias('rank')))
        
        delta = (ranks.alias('old').join(new_ranks.alias('new'), on='id')
            .select(F.abs(F.col('new.rank') - F.col('old.rank')).alias('diff'))
            .agg(F.max('diff').alias('max_diff')).collect()[0]['max_diff'])
        
        ranks = new_ranks
        ranks.cache()
        ranks.count()

        # On calcule le nouveau DataFrame
        ranks = new_ranks

        # On casse l'historique d'exécution (lineage) en faisant un checkpoint.
        # localCheckpoint() écrit les données dans les exécuteurs et tronque le DAG.
        ranks = ranks.localCheckpoint()

        # Clean up old ranks to free memory

        if iteration > 0:

            try:

                old_ranks.unpersist()

            except:

                pass

        old_ranks = ranks

        iter_time = time.time() - iter_start
        
        metrics.append({'iteration': iteration + 1, 'elapsed_sec': iter_time,
                       'max_delta': delta, 'converged': delta < tol})
        print(f'Iteration {iteration + 1:2d}: time={iter_time:.3f}s, delta={delta:.6f}')
        
        if delta < tol:
            print(f'Converged after {iteration + 1} iterations')
            break
    
    result = ranks.join(vertices_df, on='id').select('id', 'name', 'rank').orderBy(F.desc('rank'))
    return result, metrics

In [6]:
print('='*70)
print('Running PageRank with DEFAULT partitioning')
print('='*70)
pagerank_result, pagerank_metrics = run_pagerank(vertices, edges, max_iter=20, damping=0.85, tol=0.0001)
pagerank_result.cache()
print('\nTop 20 teams by PageRank:')
pagerank_result.show(20)

"""
================================================================================
FIX : GESTION DE LA MÉMOIRE SUR LES BOUCLES ITÉRATIVES SPARK (PageRank)
================================================================================
PROBLÈME RENCONTRÉ :
- Erreur : `java.lang.OutOfMemoryError: Java heap space` sur le .collect()
- Cause : Dans les algorithmes itératifs, Spark accumule l'historique de toutes 
  les transformations (le plan d'exécution appelé "DAG" ou "Lineage"). Au fil 
  des itérations, ce graphe devient gigantesque et sature la mémoire du Driver 
  lorsqu'une action (.collect, .show) est déclenchée. 
  Note : L'utilisation de .cache() ne supprime pas cet historique.

SOLUTIONS APPLIQUÉES :
1. Casser la lignée d'exécution (Lineage) :
   Remplacement de `ranks.cache()` par `ranks = ranks.localCheckpoint()` à la 
   fin de chaque itération. Le checkpoint force l'évaluation, matérialise le 
   DataFrame et, surtout, tronque le plan d'exécution précédent.
   (NB : En production sur un cluster, préférer .checkpoint() après avoir 
   défini un spark.sparkContext.setCheckpointDir()).

2. Augmenter la mémoire du Driver :
   Si le graphe de données est très volumineux, allouer plus de RAM à la 
   création de la session Spark :
   spark = SparkSession.builder \
       .config("spark.driver.memory", "4g") \
       .getOrCreate()
================================================================================
"""

Running PageRank with DEFAULT partitioning
Iteration  1: time=2.190s, delta=0.009744
Iteration  2: time=1.935s, delta=0.010221
Iteration  3: time=1.459s, delta=0.010678
Iteration  4: time=1.059s, delta=0.004814
Iteration  5: time=0.914s, delta=0.003195
Iteration  6: time=0.839s, delta=0.003608
Iteration  7: time=0.912s, delta=0.003729
Iteration  8: time=0.832s, delta=0.002254
Iteration  9: time=0.717s, delta=0.001025
Iteration 10: time=0.745s, delta=0.000377
Iteration 11: time=0.759s, delta=0.000129
Iteration 12: time=0.642s, delta=0.000035
Converged after 12 iterations

Top 20 teams by PageRank:
+----+--------------+--------------------+
|  id|          name|                rank|
+----+--------------+--------------------+
|1549|x6tence Galaxy|0.012644393935329729|
|1547|         x-kom| 0.01043873906470314|
|1516|        pro100| 0.00847193399915081|
|1502|   mousesports| 0.00608230659563646|
|1530|        subtLe|0.005094650764454578|
|1541|       vitrioL|0.004710196151004143|
|1462|   

'\n================================================================================\nFIX : GESTION DE LA MÉMOIRE SUR LES BOUCLES ITÉRATIVES SPARK (PageRank)\n================================================================================\nPROBLÈME RENCONTRÉ :\n- Erreur : `java.lang.OutOfMemoryError: Java heap space` sur le .collect()\n- Cause : Dans les algorithmes itératifs, Spark accumule l\'historique de toutes \n  les transformations (le plan d\'exécution appelé "DAG" ou "Lineage"). Au fil \n  des itérations, ce graphe devient gigantesque et sature la mémoire du Driver \n  lorsqu\'une action (.collect, .show) est déclenchée. \n  Note : L\'utilisation de .cache() ne supprime pas cet historique.\n\nSOLUTIONS APPLIQUÉES :\n1. Casser la lignée d\'exécution (Lineage) :\n   Remplacement de `ranks.cache()` par `ranks = ranks.localCheckpoint()` à la \n   fin de chaque itération. Le checkpoint force l\'évaluation, matérialise le \n   DataFrame et, surtout, tronque le plan d\'exécution préc

## Step 3 — Partitioning Experiment

In [7]:
print('='*70)
print('Running PageRank with CUSTOM partitioning')
print('='*70)
pagerank_result_custom, pagerank_metrics_custom = run_pagerank(
    vertices, edges, max_iter=20, damping=0.85, tol=0.0001, partition_strategy='custom')
pagerank_result_custom.cache()
print('\nTop 20 teams by PageRank (custom):')
pagerank_result_custom.show(20)

total_time_default = sum(m['elapsed_sec'] for m in pagerank_metrics)
total_time_custom = sum(m['elapsed_sec'] for m in pagerank_metrics_custom)
speedup = (total_time_default - total_time_custom) / total_time_default * 100

print(f'\nDefault: {total_time_default:.3f}s, Custom: {total_time_custom:.3f}s')
print(f'Speedup: {speedup:+.1f}%')

Running PageRank with CUSTOM partitioning
Applied custom partitioning: repartition(8, src)
Iteration  1: time=1.017s, delta=0.009744
Iteration  2: time=1.190s, delta=0.010221
Iteration  3: time=0.795s, delta=0.010678
Iteration  4: time=0.824s, delta=0.004814
Iteration  5: time=0.706s, delta=0.003195
Iteration  6: time=0.785s, delta=0.003608
Iteration  7: time=0.840s, delta=0.003729
Iteration  8: time=0.875s, delta=0.002254
Iteration  9: time=0.688s, delta=0.001025
Iteration 10: time=0.791s, delta=0.000377
Iteration 11: time=0.741s, delta=0.000129
Iteration 12: time=0.853s, delta=0.000035
Converged after 12 iterations

Top 20 teams by PageRank (custom):
+----+--------------+--------------------+
|  id|          name|                rank|
+----+--------------+--------------------+
|1549|x6tence Galaxy|0.012644393935329729|
|1547|         x-kom| 0.01043873906470314|
|1516|        pro100| 0.00847193399915081|
|1502|   mousesports|0.006082306595636...|
|1530|        subtLe|0.005094650764454

## Step 4 — Convergence Analysis

In [8]:
%pip install "numpy<2"

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [9]:
# Install matplotlib if not available

try:

    import matplotlib.pyplot as plt

except ImportError:

    import subprocess

    import sys

    print('Installing matplotlib...')

    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'matplotlib'])

    import matplotlib.pyplot as plt

    print('matplotlib installed successfully!')



import pandas as pd

df_default = pd.DataFrame(pagerank_metrics)
df_custom = pd.DataFrame(pagerank_metrics_custom)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Elapsed time
axes[0, 0].plot(df_default['iteration'], df_default['elapsed_sec'], marker='o', label='Default')
axes[0, 0].plot(df_custom['iteration'], df_custom['elapsed_sec'], marker='s', label='Custom')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].set_ylabel('Time (s)')
axes[0, 0].set_title('Per-Iteration Time')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Convergence delta
axes[0, 1].semilogy(df_default['iteration'], df_default['max_delta'], marker='o', label='Default')
axes[0, 1].semilogy(df_custom['iteration'], df_custom['max_delta'], marker='s', label='Custom')
axes[0, 1].axhline(y=0.0001, color='r', linestyle='--', label='Tolerance')
axes[0, 1].set_xlabel('Iteration')
axes[0, 1].set_ylabel('Max Delta (log)')
axes[0, 1].set_title('Convergence Delta')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Cumulative time
df_default['cumulative_time'] = df_default['elapsed_sec'].cumsum()
df_custom['cumulative_time'] = df_custom['elapsed_sec'].cumsum()
axes[1, 0].plot(df_default['iteration'], df_default['cumulative_time'], marker='o', label='Default')
axes[1, 0].plot(df_custom['iteration'], df_custom['cumulative_time'], marker='s', label='Custom')
axes[1, 0].set_xlabel('Iteration')
axes[1, 0].set_ylabel('Cumulative Time (s)')
axes[1, 0].set_title('Cumulative Time')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Bar comparison
x = range(1, min(len(df_default), len(df_custom)) + 1)
width = 0.35
axes[1, 1].bar([i - width/2 for i in x], df_default['elapsed_sec'][:len(x)], width, label='Default')
axes[1, 1].bar([i + width/2 for i in x], df_custom['elapsed_sec'][:len(x)], width, label='Custom')
axes[1, 1].set_xlabel('Iteration')
axes[1, 1].set_ylabel('Time (s)')
axes[1, 1].set_title('Time Comparison')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(OUT_DIR / 'convergence_analysis.png', dpi=150)
print(f'Saved: {OUT_DIR / "convergence_analysis.png"}')
plt.show()


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/home/florian/.local/lib/python3.12/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/home/florian/.local/lib/python3.12/site-packages/ipykernel/kernelapp.py", line 758, in start
    self.io_loop.start()
  File "/home/florian/.lo

ImportError: 
A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.4 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.



Installing matplotlib...


error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'matplotlib']' returned non-zero exit status 1.

## Step 5 — Evidence & Metrics

In [ ]:
# Save execution plans
with open(PROOF_DIR / 'plan_graph.txt', 'w') as f:
    f.write('GRAPH CONSTRUCTION PLAN\n' + '='*70 + '\n')
edges.explain(mode='extended')

with open(PROOF_DIR / 'plan_pagerank.txt', 'w') as f:
    f.write('PAGERANK RESULT PLAN\n' + '='*70 + '\n')
pagerank_result.explain(mode='extended')

# Export results
top_teams = pagerank_result.limit(50).toPandas()
top_teams.to_csv(OUT_DIR / 'top_teams.csv', index=False)
df_default.to_csv(OUT_DIR / 'metrics_default.csv', index=False)
df_custom.to_csv(OUT_DIR / 'metrics_custom.csv', index=False)

print(f'Saved: {OUT_DIR / "top_teams.csv"}')
print(f'Saved: {OUT_DIR / "metrics_default.csv"}')
print(f'Saved: {OUT_DIR / "metrics_custom.csv"}')

PySparkValueError: [CANNOT_SET_TOGETHER] extended and mode should not be set together.

In [ ]:
import csv

metrics_data = [
    ['metric', 'value', 'unit', 'description'],
    ['graph_vertices', n_vertices, 'count', 'Unique teams'],
    ['graph_edges', n_edges, 'count', 'Team pairs'],
    ['graph_density', f'{n_edges / (n_vertices * (n_vertices - 1) / 2):.6f}', 'ratio', 'Density'],
    ['pagerank_damping', 0.85, 'ratio', 'Damping factor'],
    ['pagerank_tolerance', 0.0001, 'threshold', 'Convergence tolerance'],
    ['default_iterations', len(df_default), 'count', 'Iterations (default)'],
    ['default_total_time', f'{df_default["elapsed_sec"].sum():.3f}', 'seconds', 'Total time (default)'],
    ['default_mean_time', f'{df_default["elapsed_sec"].mean():.3f}', 'seconds', 'Mean time (default)'],
    ['custom_iterations', len(df_custom), 'count', 'Iterations (custom)'],
    ['custom_total_time', f'{df_custom["elapsed_sec"].sum():.3f}', 'seconds', 'Total time (custom)'],
    ['custom_mean_time', f'{df_custom["elapsed_sec"].mean():.3f}', 'seconds', 'Mean time (custom)'],
    ['speedup_percent', f'{speedup:.2f}', 'percent', 'Speedup'],
]

for i, row in enumerate(top_teams.head(10).itertuples(), 1):
    metrics_data.append([f'top_team_{i}', row.name, f'{row.rank:.6f}', f'Rank #{i}'])

with open(METRICS_LOG, 'w', newline='', encoding='utf-8') as f:
    csv.writer(f).writerows(metrics_data)

print(f'Metrics log: {METRICS_LOG}')
print(f'Total metrics: {len(metrics_data) - 1}')
print(f'\nTop 3 teams:')
for i, row in enumerate(top_teams.head(3).itertuples(), 1):
    print(f'  {i}. {row.name}: {row.rank:.6f}')

NameError: name 'df_default' is not defined

In [10]:
spark.stop()
print('Done.')

Done.
